# OsteoLab — flujo de ML de principio a fin

Un solo cuaderno con el recorrido completo del modelo: de dónde salen las imágenes, cómo se
miden, qué filtros se aplican, cómo se entrena cada variante, cómo se evalúa de forma honesta y
cómo se sirve en inferencia.

El cuaderno **no reimplementa nada**: importa y ejecuta los mismos módulos que usan la API y los
scripts de entrenamiento (`shared/`, `training/`), así que lo que se ve aquí es exactamente lo que
corre en producción.

Contexto y decisiones históricas: `README.md` (estado actual) y `base_datos_osea/HISTORIAL.md`
(cronología de por qué cada cosa es como es).

**Índice**

| Sección | Qué cubre |
|---|---|
| 1 | Setup y mapa del repositorio |
| 2 | Datos: mallas, renders y fotos reales |
| 3 | Features geométricas con OpenCV (una imagen, paso a paso) |
| 4 | Construcción de las tablas de entrenamiento |
| 5 | Exploración del dataset (clases, PCA, separabilidad) |
| 6 | Filtros: CLIP zero-shot, ResNet18 binario, segmentadores |
| 7 | Entrenamiento del modelo geométrico (Random Forest) |
| 8 | Entrenamiento CNN / híbrido |
| 9 | Evaluación honesta: banco sintético y fotos reales |
| 10 | Cascada de 2 etapas (hueso fino y taxón) |
| 11 | Inferencia end-to-end y API |
| 12 | Reproducibilidad: DVC, MLflow, tests |

## 1. Setup y mapa del repositorio

El cuaderno se ejecuta desde la raíz del repo con el entorno del proyecto:

```bash
source .venv/bin/activate
pip install -r requirements.txt -r requirements-notebook.txt   # extras: jupyterlab, ipykernel, matplotlib
jupyter lab notebooks/osteolab_pipeline.ipynb
```
 Las celdas que entrenan o
regeneran datasets están desactivadas por defecto con los interruptores `RUN_*`: al ponerlos en
`True` lanzan el script real como subproceso y muestran su salida.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def find_repo_root():
    """Sube desde el directorio de trabajo (o desde el del propio notebook) hasta
    encontrar `params.yaml`: así da igual si el kernel arranca en la raíz del repo,
    en `notebooks/` o en cualquier subcarpeta."""
    starts = [Path.cwd()]
    nb_file = globals().get("__vsc_ipynb_file__") or globals().get("__session__")
    if nb_file:
        starts.insert(0, Path(nb_file).resolve().parent)
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / "params.yaml").exists() and (candidate / "training").is_dir():
                return candidate
    raise RuntimeError(
        "No encuentro la raíz del repo (params.yaml). Abre el notebook desde "
        "osteolab-ml-platform/ o define REPO_ROOT a mano en esta celda."
    )


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Interruptores de las celdas caras (horas de CPU/GPU). Dejar en False para leer el flujo.
RUN_BUILD_DATASETS = False   # sección 4: re-extraer features de todos los renders
RUN_TRAIN_GEOMETRIC = False  # sección 7: Random Forest (minutos)
RUN_TRAIN_CNN = False        # sección 8: fine-tuning del backbone (decenas de minutos con GPU)
RUN_EVALUATE = False         # sección 9: evaluación sobre bancos externos

PROCESSED = REPO_ROOT / "data/processed"
MODELS = REPO_ROOT / "models"
BONE_DB = REPO_ROOT / "base_datos_osea"

print("repo:", REPO_ROOT)


def run(cmd, enabled, cwd=REPO_ROOT):
    """Lanza un script del repo y muestra su salida; si `enabled` es False solo imprime el comando."""
    printable = " ".join(str(c) for c in cmd)
    if not enabled:
        print(f"[desactivado] {printable}")
        return None
    print(f"$ {printable}\n")
    proc = subprocess.run([sys.executable, *cmd[1:]] if cmd[0] == "python" else cmd,
                          cwd=cwd, text=True, capture_output=True)
    print(proc.stdout[-8000:])
    if proc.returncode != 0:
        print("STDERR:\n", proc.stderr[-4000:])
    return proc

### Piezas del flujo

| Capa | Código | Artefacto |
|---|---|---|
| Catálogo y descarga de mallas 3D | `base_datos_osea/scripts/`, `shared/bone_ingest/` | `base_datos_osea/data/meshes/` |
| Render multi-vista (Blender / BlenderProc) | `base_datos_osea/blender/`, `shared/bone_ingest/blender_split_render.py` | `base_datos_osea/renders*/` |
| Medida geométrica (OpenCV + segmentador) | `shared/cv_extractor/extract.py` | dict de features por imagen |
| Tablas de entrenamiento | `shared/cv_extractor/build_dataset*.py` | `data/processed/*.csv` |
| Filtro "¿es un hueso?" | `shared/clip_filter/`, `shared/bone_filter/` | `models/bone_filter/` |
| Modelo geométrico | `training/train_geometric.py` | `models/geometric_*.joblib` |
| Modelo CNN / híbrido | `training/train_cnn.py` | `models/cnn_*.{pt,joblib}` |
| Evaluación | `training/evaluate*.py` | métricas en MLflow + `reports/` |
| Inferencia | `shared/inference/predict_*.py` | API FastAPI (`serving/backend/main.py`) |

In [ ]:
for sub in ["shared/cv_extractor", "shared/inference", "training", "shared/clip_filter", "shared/bone_filter"]:
    files = sorted(p.name for p in (REPO_ROOT / sub).glob("*.py"))
    print(f"{sub:26s} {', '.join(files)}")

## 2. Datos: mallas, renders y fotos reales

No hay un `data/raw/<clase>/` con fotos: el dataset base es **sintético**. Cada hueso es una malla
3D verificada a mano (MorphoSource / Sketchfab / Smithsonian, ver `HISTORIAL.md`) renderizada en 24
vistas. Sobre esa base hay varias **variantes de render**, que existen porque la silueta limpia sola
no generalizaba a fotos reales:

| Variante | Carpeta | Qué añade |
|---|---|---|
| `renders` | `base_datos_osea/renders/` | silueta plana, fondo uniforme — el punto de partida |
| `renders_noisy_composited` | `.../renders_noisy_composited/` | el hueso compuesto sobre fondos reales |
| `renders_noisy_bproc` | `.../renders_noisy_bproc/` | BlenderProc: luz, sombras y materiales físicos |
| `renders_lab` | `.../renders_lab/` | escena de laboratorio (mesa, HDRI) — **la que entra en el modelo por defecto** |
| Fotos reales | `base_datos_osea/Fotos entrenamiento/_curated/` | fotos de hueso real, curadas a mano |

La **taxonomía es de 9 grupos morfológicos** (`cranio`, `mandibula_maxilar`, `hueso_largo`,
`hueso_plano`, `pelvis`, `sacro`, `vertebra`, `costilla`, `hueso_pequeno`) y no de hueso exacto ni
de especie: con 1-9 especímenes físicos por hueso, un clasificador de 21 huesos finos solo aprende
a reconocer *ese objeto concreto* (comprobado con `GroupKFold` dejando especímenes fuera).

In [ ]:
for d in sorted(BONE_DB.glob("renders*")):
    n = sum(1 for _ in d.rglob("*.png")) + sum(1 for _ in d.rglob("*.jpg"))
    print(f"{d.name:32s} {n:6d} imágenes")

curated = BONE_DB / "Fotos entrenamiento" / "_curated"
if curated.exists():
    print("\nfotos reales curadas:")
    for d in sorted(p for p in curated.iterdir() if p.is_dir()):
        print(f"  {d.name:12s} {len(list(d.glob('*')))}")

### Criterio de curación de fotos reales

Se filtra por **grupo morfológico, no por especie**: una mandíbula de perro cuenta igual que una
humana, porque la clase que se aprende es la forma del hueso. Lo que descalifica una foto es que el
hueso no se distinga del fondo, que esté fragmentado, o que no se corresponda con el grupo
etiquetado. El manifiesto de lo aceptado vive en `data/real_photos_accepted/manifest.csv` y se
revisa con `scripts/review_app.py`.

## 3. Features geométricas con OpenCV (una imagen, paso a paso)

`shared/cv_extractor/extract.py::extract_features` es la única fuente de medidas de todo el
proyecto: la usan la construcción del dataset, el modelo híbrido y la API. No hay aprendizaje en
este paso — es segmentación clásica más geometría de contornos.

Qué devuelve:

- **Geometría base**: `area_px`, `area_ratio`, `aspect_ratio`, `extent`, `solidity`,
  `bounding_box_px`, `rotated_box_px` (alineado con el eje del hueso, no con la imagen).
- **Forma**: `elongation` (PCA del contorno), `ellipse_ratio`, `circularity`, `convexity`,
  `hu_moments` (7 momentos de Hu con la transformación logarítmica).
- **Perfil de anchura**: `width_profile` (20 cortes perpendiculares al eje) y sus estadísticos.
- **Curvatura**: `curvature_mean`/`curvature_max` de la línea central (curvatura de Menger).
- **Visual**: `annotated_image_base64`, `mask_overlay_base64`, y `crop_image` si se pide
  `include_crop=True` (el recorte con fondo gris que ve la CNN).

Todas las medidas son relativas a la imagen (píxeles y ratios). Sin una referencia de escala en la
foto no hay forma de derivar centímetros, así que no se intenta.

In [ ]:
import base64
import io

from IPython.display import HTML, display

from shared.cv_extractor.extract import extract_features

sample = next(iter(sorted((BONE_DB / "renders_lab").rglob("*.png"))), None) \
    or next(iter(sorted(curated.rglob("*.jpg"))))
print("imagen de ejemplo:", sample.relative_to(REPO_ROOT))

with open(sample, "rb") as fh:
    feats = extract_features(fh, backend="classic")   # "grounded_sam"/"fastsam" para fotos reales

print({k: v for k, v in feats.items() if not k.endswith("_base64") and k != "crop_image"})

In [ ]:
# Verificación visual de la segmentación: medidas dibujadas + máscara detectada.
display(HTML("".join(
    f'<img src="data:image/png;base64,{feats[k]}" style="max-width:45%;margin:4px">'
    for k in ("annotated_image_base64", "mask_overlay_base64") if feats.get(k)
)))

### Backends de segmentación

El contorno se puede obtener de tres formas, y la elección importa mucho más en fotos reales que en
renders:

| Backend | Cómo funciona | Cuándo |
|---|---|---|
| `classic` | Otsu + contornos (sin aprendizaje) | renders con fondo limpio |
| `fastsam` | FastSAM (`models/FastSAM-s.pt`) | alternativa rápida sobre fotos |
| `grounded_sam` | prompt de texto + SAM | **default en fotos reales**, el más robusto a sombras y textura |

Una segmentación sucia no solo degrada la clasificación: rompe las features que dependen de un
contorno fino (contar agujeros, circularidad exacta) — por eso la cascada de la sección 10 solo se
fía de `elongation`, que mide la envolvente general.

In [ ]:
# Mismo recorte que ve la CNN (fondo gris fuera de la máscara).
with open(sample, "rb") as fh:
    with_crop = extract_features(fh, backend="classic", include_crop=True)
if with_crop.get("crop_image") is not None:
    import numpy as np
    from PIL import Image
    crop = with_crop["crop_image"]
    print("recorte:", getattr(crop, "shape", type(crop)))
    display(Image.fromarray(np.asarray(crop)[:, :, ::-1]) if getattr(crop, "ndim", 0) == 3 else crop)

## 4. Construcción de las tablas de entrenamiento

Cada constructor recorre una carpeta de renders, mide cada imagen con `extract_features`, aplana el
dict a columnas (`flatten_features`) y escribe un CSV con la etiqueta deducida de la ruta
(`renders/<especie>/<hueso>/<espécimen>/vista.png`).

Clave metodológica: **unos especímenes completos se reservan como banco** (`BENCHMARK_SPECIMENS` en
`build_dataset.py`) y se escriben en un CSV aparte, nunca en el de entrenamiento. Si se dejaran
dentro, las 24 vistas del mismo objeto se repartirían entre train y test y la métrica saldría
inflada.

In [ ]:
run(["python", "-m", "shared.cv_extractor.build_dataset"], RUN_BUILD_DATASETS)            # renders/
run(["python", "-m", "shared.cv_extractor.build_dataset_lab"], RUN_BUILD_DATASETS)        # renders_lab/
run(["python", "-m", "shared.cv_extractor.build_dataset_bproc"], RUN_BUILD_DATASETS)      # renders_noisy_bproc/
run(["python", "-m", "shared.cv_extractor.build_dataset_real_photos", "--split", "train"], RUN_BUILD_DATASETS)
run(["python", "-m", "shared.cv_extractor.build_dataset_real_photos", "--split", "test"], RUN_BUILD_DATASETS)
run(["python", "scripts/build_combined_lab_variant.py"], RUN_BUILD_DATASETS)              # receta combinada

La tabla que alimenta el modelo por defecto es la **combinada**: `renders_lab` +
`noisy_composited` + `noisy_bproc` + fotos reales (split train). Es la receta `hybrid_lab`, promovida
a default tras confirmarse con 3 semillas.

In [ ]:
import pandas as pd

for name in ["bone_geometric_features.csv", "bone_geometric_features_lab.csv",
             "bone_geometric_features_combined.csv", "bone_geometric_features_combined_lab.csv",
             "bone_geometric_benchmark.csv", "bone_geometric_features_real_photos_test.csv"]:
    path = PROCESSED / name
    if path.exists():
        df = pd.read_csv(path)
        print(f"{name:48s} {len(df):6d} filas  {len(df.columns):3d} columnas")

## 5. Exploración del dataset

In [ ]:
dataset_path = PROCESSED / "bone_geometric_features_combined_lab.csv"
if not dataset_path.exists():
    dataset_path = PROCESSED / "bone_geometric_features.csv"
df = pd.read_csv(dataset_path)
print(dataset_path.name, df.shape)
df.groupby("bone_group").size().sort_values(ascending=False)

In [ ]:
# Procedencia de cada fila (render sintético vs foto real) y especímenes distintos por clase.
meta_cols = [c for c in ["bone_group", "bone", "species", "specimen", "source", "image_path"] if c in df.columns]
print("columnas de metadatos:", meta_cols)
if "specimen" in df.columns:
    print(df.groupby("bone_group")["specimen"].nunique().to_string())

In [ ]:
import matplotlib.pyplot as plt

num_cols = [c for c in df.select_dtypes("number").columns if c not in {"area_px"}]
interesting = [c for c in ["elongation", "aspect_ratio", "circularity", "solidity", "convexity",
                           "curvature_mean", "mean_width", "std_width"] if c in df.columns]
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, col in zip(axes.ravel(), interesting):
    for group, sub in df.groupby("bone_group"):
        ax.hist(sub[col].dropna(), bins=25, alpha=0.4, label=group)
    ax.set_title(col, fontsize=9)
axes.ravel()[0].legend(fontsize=6)
plt.tight_layout()
plt.show()

`elongation` es la única feature con una frontera limpia y que **además** sobrevive a fotos reales
(`hueso_largo` frente al resto). Las demás separan bien en el banco sintético limpio y se derrumban
en foto real: es la observación que justifica la cascada de la sección 10.

In [ ]:
# Espacio PCA de referencia: el mismo que la API devuelve en GET /pca/reference.
ref_path = PROCESSED / "bone_geometric_pca_reference.csv"
if ref_path.exists():
    ref = pd.read_csv(ref_path)
    fig, ax = plt.subplots(figsize=(7, 6))
    for group, sub in ref.groupby("bone_group"):
        ax.scatter(sub["pca_x"], sub["pca_y"], s=8, alpha=0.6, label=group)
    ax.legend(fontsize=7)
    ax.set_title("PCA (2 componentes) sobre las features estandarizadas")
    plt.show()
else:
    print("falta el PCA de referencia — se genera al entrenar (sección 7)")

## 6. Filtros antes de clasificar

### 6.1 CLIP zero-shot — ¿es un hueso?

Primera puerta del pipeline y sin entrenamiento propio: compara el embedding de la imagen contra
descripciones de texto fijas (`shared/clip_filter/model.py`: `BONE_DESCRIPTIONS`,
`NOT_BONE_DESCRIPTIONS`, `BONE_TYPE_DESCRIPTIONS`), softmax conjunta y suma por grupo. Si dice que
no es un hueso, el flujo se detiene ahí y no se gasta ni segmentación ni CNN. Si supera
`SUGGESTION_CERTAIN_THRESHOLD` (0.7), además propone tipo con `certain: true`.

In [ ]:
from shared.clip_filter.predict import classify_image as clip_classify

with open(sample, "rb") as fh:
    print(clip_classify(fh))

### 6.2 Filtro binario ResNet18 (endpoint suelto)

Filtro entrenado con datos propios (`data/raw/bone_filter/bone|not_bone/`). Sigue expuesto en
`POST /filter/is-bone` pero **ya no está en el flujo por defecto** — CLIP cubre la puerta de entrada
sin necesitar dataset. Se mantiene aislado: no comparte código ni estado con el resto.

In [ ]:
run(["python", "shared/bone_filter/train.py", "--epochs", "15", "--batch-size", "64", "--lr", "1e-4"],
    enabled=False)   # pesos en models/bone_filter/bone_filter_resnet18.pt + labels.json

### 6.3 El filtro que de verdad decide: la segmentación

En la práctica el "filtro" determinante no es binario sino la calidad del recorte: desde el
2026-07-26 **la CNN nunca ve la imagen completa**, solo el recorte al bounding box con el fondo
pintado de gris (`bg_gray=128`). Entrenamiento e inferencia comparten backend de segmentación —
si no coincidieran, el modelo vería en producción una distribución distinta a la que aprendió.

## 7. Modelo geométrico (Random Forest)

Entrena sobre las features numéricas de la tabla, según `params.yaml`
(`geometric_training`, `geometric_model`), y guarda cuatro artefactos más el PCA de referencia.
El orden exacto de columnas se persiste (`geometric_feature_columns_bone_group.joblib`) porque un
Random Forest solo conoce la posición de cada columna, no su significado: si en inferencia el orden
no coincide, las probabilidades salen sin sentido.

Se evalúa **dos veces**:

1. split aleatorio por fila — optimista, deja fugarse vistas del mismo espécimen entre train y test;
2. `GroupKFold` repetido 10 veces con **especímenes completos fuera** — la lectura honesta.

Ambas se imprimen y se registran en MLflow.

In [ ]:
import yaml

print(yaml.safe_load((REPO_ROOT / "params.yaml").read_text()))

In [ ]:
run(["python", "-m", "training.train_geometric"], RUN_TRAIN_GEOMETRIC)
# --target bone entrena el clasificador de 21 huesos finos (comparativa, no producción)

In [ ]:
# Los splits honestos son código aparte y con test propio (tests/test_group_splits.py).
import inspect

from training import group_splits

print(inspect.getsource(group_splits.subject_ids))

In [ ]:
import joblib

model_files = sorted(p.name for p in MODELS.glob("geometric_*"))
print(model_files)
if (MODELS / "geometric_model_bone_group.joblib").exists():
    rf = joblib.load(MODELS / "geometric_model_bone_group.joblib")
    cols = joblib.load(MODELS / "geometric_feature_columns_bone_group.joblib")
    imp = pd.Series(rf.feature_importances_, index=cols).sort_values(ascending=False)
    print(imp.head(15).to_string())

## 8. Modelo CNN / híbrido

`training/train_cnn.py` entrena tres variantes sobre el mismo recorte segmentado:

| `--model-type` | Entrada | Idea |
|---|---|---|
| `cnn_only` | solo el recorte | backbone `timm` (`resnet18` por defecto) con cabeza nueva |
| `hybrid` | recorte + vector geométrico | **el default de producción** (`hybrid_lab`) |
| `gated_hybrid` | recorte + geometría con puerta aprendida | pondera ambas ramas por imagen |

Receta de entrenamiento: `--epochs-frozen 5` con el backbone congelado (solo cabeza, `lr 1e-3`),
luego `--epochs-finetune 10` descongelando (`--lr-backbone 1e-5`). Las fotos reales pesan más en la
pérdida (`--real-photo-boost 4.0`) porque son pocas y son el dominio que importa, y
`--select-by-real-bench` elige el checkpoint por el banco de fotos reales en vez de por la
validación sintética.

In [ ]:
run(["python", "-m", "training.train_cnn",
     "--model-type", "hybrid",
     "--dataset-path", "data/processed/bone_geometric_features_combined_lab.csv",
     "--run-name", "hybrid_lab",
     "--epochs-frozen", "5", "--epochs-finetune", "10",
     "--real-photo-boost", "4.0",
     "--select-by-real-bench"], RUN_TRAIN_CNN)

### Varianza entre semillas

Con un dataset de este tamaño, una sola corrida no distingue una mejora real del ruido de
inicialización. Un cambio se acepta cuando se mantiene en **3 semillas** (`--model-seed`): así se
confirmó `hybrid_lab` (0.8701 / 0.8323) antes de promoverlo a default.

In [ ]:
for seed in (0, 1, 2):
    run(["python", "-m", "training.train_cnn",
         "--model-type", "hybrid",
         "--dataset-path", "data/processed/bone_geometric_features_combined_lab.csv",
         "--run-name", f"hybrid_lab_seed{seed}",
         "--model-seed", str(seed),
         "--select-by-real-bench"], RUN_TRAIN_CNN)

In [ ]:
print(sorted(p.name for p in MODELS.glob("cnn_*hybrid*")))

## 9. Evaluación honesta

Tres bancos, de más fácil a más difícil, y lo único que cuenta como resultado del proyecto es el
último:

1. **validación interna** (split por filas) — útil solo para detectar que algo está roto;
2. **banco sintético excluido** (`bone_geometric_benchmark.csv`) — especímenes completos que el
   modelo no ha visto nunca;
3. **banco de fotos reales** — otro dominio, no solo otro objeto: la prueba que de verdad importa.

In [ ]:
run(["python", "-m", "training.evaluate"], RUN_EVALUATE)                                  # geométrico
run(["python", "-m", "training.evaluate_cnn", "--model-type", "hybrid"], RUN_EVALUATE)    # CNN/híbrido
run(["python", "-m", "training.evaluate_real_photos", "--segmenter", "grounded_sam"], RUN_EVALUATE)
run(["python", "-m", "training.evaluate_cnn_real_photos",
     "--model-type", "hybrid", "--segmenter", "grounded_sam", "--source", "benchmark"], RUN_EVALUATE)

In [ ]:
# Informes ya generados (matrices de confusión, curvas ROC, métricas por clase).
for p in sorted((REPO_ROOT / "reports").rglob("*")):
    if p.is_file():
        print(p.relative_to(REPO_ROOT))

In [ ]:
run(["python", "-m", "training.plot_roc_curves"], RUN_EVALUATE)
run(["python", "-m", "training.plot_roc_cnn"], RUN_EVALUATE)

Lectura de los resultados: el agregado no basta, hay que mirar por clase. `hueso_plano` sigue
siendo la clase débil (pocos especímenes y una forma que solapa con `cranio` y `pelvis` una vez
segmentada). En el piloto de taxón, `ungulado` en `cranio` pasó de 0.0 a 0.6 de recall solo al
sumar fotos reales de ese grupo — con tan pocas clases por rama, añadir datos del dominio real
mueve la métrica más que cualquier cambio de arquitectura.

## 10. Cascada de 2 etapas

### 10.1 Puerta geométrica para `hueso_largo`

Regla barata y de alta precisión antes de la CNN: si `elongation > 0.900` (umbral de un stump de
profundidad 1 ajustado **sobre datos de entrenamiento**), la respuesta es `hueso_largo` y no se
ejecuta el modelo. Se probaron otras puertas (`n_holes` para sacro/vértebra, `circularity`/
`solidity` para cráneo/plano): todas parecían equivalentes en el banco sintético (88-92%) y solo
`elongation` aguantó en fotos reales (93%, 1 falso positivo en 47); el resto bajó hasta 16% con 46
falsos positivos en 49. Antes de añadir una regla nueva aquí hay que validarla igual: accuracy
**y** falsos positivos en el banco de fotos reales.

In [ ]:
from shared.inference.predict_cascade import ELONGATION_THRESHOLD, classify_image as cascade_classify

print("umbral de elongación:", ELONGATION_THRESHOLD)
with open(sample, "rb") as fh:
    out = cascade_classify(fh, model_type="cnn_only", backend="classic")
print({k: v for k, v in out.items() if k != "cv_features"})   # "gate": elongation_rule | cnn

### 10.2 Etapa 2: hueso concreto y taxón

Entrenar un único modelo con todas las clases finas hacía que ramas no relacionadas interfirieran
(`hueso_plano` colapsaba). La solución en producción es una cascada: la etapa 1 da el grupo, y según
el grupo se invoca un modelo especializado.

| Grupo de etapa 1 | Modelo de etapa 2 | Qué afina |
|---|---|---|
| `hueso_largo` | `stage2_hueso_largo` | fémur / húmero / … |
| `hueso_pequeno` | `stage2_hueso_pequeno` | hueso fino de mano/pie |
| `vertebra` | `stage2_vertebra` | cervical / torácica / lumbar |
| `cranio` | `stage2_taxon_cranio` | taxón (humano / carnívoro / ungulado / roedor / otro primate) |
| `mandibula_maxilar` | `stage2_taxon_mandibula_maxilar` | taxón |

Los grupos sin modelo de etapa 2 (`costilla`, `hueso_plano`, `pelvis`, `sacro`) devuelven
`stage2_pending: true` — distingue explícitamente "aquí no aplica" de "falta entrenar esto".

In [ ]:
from shared.inference.predict_two_stage import (STAGE2_MODELS, STAGE2_TAXON_MODELS,
                                                classify_image_two_stage)

print("hueso fino:", STAGE2_MODELS)
print("taxón:    ", STAGE2_TAXON_MODELS)
with open(sample, "rb") as fh:
    res = classify_image_two_stage(fh, backend="classic")
print({k: v for k, v in res.items() if k != "cv_features"})

In [ ]:
# Entrenamiento de una rama de etapa 2: misma receta, dataset y target distintos.
run(["python", "-m", "training.train_cnn",
     "--model-type", "hybrid",
     "--dataset-path", "data/processed/bone_geometric_features_stage2_vertebra.csv",
     "--target", "bone_fine",
     "--run-name", "stage2_vertebra"], RUN_TRAIN_CNN)

run(["python", "-m", "training.evaluate_cascade_real_photos",
     "--model-type", "cnn_only", "--segmenter", "grounded_sam"], RUN_EVALUATE)

## 11. Inferencia end-to-end y API

Lo que corre al subir una foto en la web (`serving/backend/static/`) o la UI Streamlit:

1. `POST /filter/clip` — ¿es un hueso? Si no, fin.
2. `POST /classify` — mide con OpenCV, clasifica con el Random Forest y proyecta en el PCA 2D.
3. `POST /hybrid/classify` — cascada de 2 etapas sobre el modelo híbrido (grupo + hueso fino/taxón
   + `stage2_pending`).
4. `GET /pca/reference` — las ~1300 vistas de fondo del scatter (cacheado con `lru_cache`).

Cada módulo de inferencia carga sus artefactos **una vez al importar** (variables de módulo, no por
request) y falla con un mensaje explícito si el modelo no está entrenado todavía.

In [ ]:
from shared.inference.predict_geometric import classify_image as geometric_classify

with open(sample, "rb") as fh:
    g = geometric_classify(fh)
print("grupo:", g.get("prediction"))
print("probabilidades:", g.get("probabilities"))
print("punto PCA:", g.get("pca_point"))

In [ ]:
# La misma imagen por las tres rutas, para comparar lado a lado.
rows = []
for name, fn, kwargs in [
    ("geométrico", geometric_classify, {}),
    ("cascada", cascade_classify, {"model_type": "cnn_only", "backend": "classic"}),
    ("2 etapas", classify_image_two_stage, {"backend": "classic"}),
]:
    with open(sample, "rb") as fh:
        try:
            out = fn(fh, **kwargs)
            rows.append({"ruta": name, "predicción": out.get("prediction"),
                         "extra": out.get("gate") or out.get("stage2_prediction")})
        except Exception as exc:                      # modelo no entrenado en este entorno
            rows.append({"ruta": name, "predicción": f"error: {type(exc).__name__}", "extra": str(exc)[:60]})
pd.DataFrame(rows)

In [ ]:
# Contra la API levantada (uvicorn serving.backend.main:app --port 8000):
print("""
curl http://localhost:8000/health
curl -X POST http://localhost:8000/classify        -F "file=@imagen.jpg"
curl -X POST http://localhost:8000/hybrid/classify -F "file=@imagen.jpg"
curl http://localhost:8000/pca/reference
""".strip())

## 12. Reproducibilidad

- **DVC** (`dvc.yaml`) declara la etapa `train_geometric` con sus dependencias, parámetros y salidas:
  `dvc repro` reconstruye el modelo si cambió el dataset, el código o `params.yaml`.
- **MLflow** recibe cada corrida con sus parámetros, métricas (ambos splits) y tags; experimentos
  separados por modelo (`osteolab-bone-classification-geometric-bone_group`,
  `osteolab-bone-filter`, …) para que no se mezclen.
- **Tests** (`pytest`): `tests/test_group_splits.py` protege que los splits no filtren especímenes
  entre train y test, `tests/test_hybrid_endpoint.py` cubre el contrato de `/hybrid/classify`.
- **Historial** (`base_datos_osea/HISTORIAL.md`): append-only, el porqué de cada decisión.

In [ ]:
run(["python", "-m", "pytest", "-q", "tests"], enabled=False)
run(["dvc", "repro"], enabled=False)
print("\nMLflow: mlflow ui --port 5000  (o http://localhost:5000 con el stack de Docker levantado)")